# 결측, 상수 컬럼과 중복 행 확인

담당: 황유진. 온도 원본의 결측 개수와 값 범위를 확인함. 이 노트북에서 값을 채우거나 행을 삭제하지 않음.


## 공통 준비

In [ ]:
import os
import pandas as pd

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
FIG_DIR = os.path.join(ROOT, "figures")
ENC = "utf-8-sig"

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")
BASE_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v1.csv")



pd.set_option("display.max_columns", 50)
print("경로 확인:", os.path.exists(TEMP_CSV), os.path.exists(EVENT_CSV), os.path.exists(REPAIR_CSV))

### 1단계 — 데이터를 읽고 전체 모양을 보세요

In [ ]:
df = pd.read_csv(TEMP_CSV, encoding=ENC)

print("행 수, 열 수:", df.shape)
df.head()

### 2단계 — 컬럼별 빈칸(결측) 개수를 세세요

`isna()` 는 빈칸이면 True를 돌려줍니다. 여기에 `.sum()` 을 붙이면 개수가 나옵니다.

In [ ]:
empty = df.isna().sum()
print(empty)

print("===============")

empty_per = (df.isna().mean() * 100).round(2)
print(empty_per)

### 3단계 — 값이 계속 똑같은 컬럼을 찾으세요

`nunique()` 는 그 컬럼에 **서로 다른 값이 몇 개 있는지** 세어줍니다.
이 값이 1이면 처음부터 끝까지 같은 값만 들어 있다는 뜻입니다.

In [ ]:
count = df.nunique()
print(count)

print("===============")

#       print(list(const_cols))
const_cols = df.columns[df.nunique() <= 1]
print(list(const_cols))

### 4단계 — 중복된 행이 있는지 확인하세요

두 가지를 봅니다.

1. **모든 컬럼이 똑같은 행**이 두 번 들어간 경우
2. **같은 시각(MEAS_DT)이 두 번 이상** 나온 경우 — 값이 달라도 문제입니다

In [ ]:
same = df.duplicated().sum()
print(same)

print("===============")

same_time = df["MEAS_DT"].duplicated().sum()
print(same_time)

### 5단계 — 숫자 컬럼의 값 범위를 훑어보세요

온도 컬럼(`TC-Z1` ~ `TC-Z6` 등)에 말이 안 되는 값(예: 음수, 0, 아주 큰 값)이 있는지 봅니다.

In [ ]:
df[["TC-Z1", "TC-Z2", "TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6"]].describe().T
# 눈에 띄는 값이 있으면 아래 결과 정리에 적어주세요.

## 확인 결과


| 항목 | 확인한 값 |
| --- | --- |
| 전체 행 수 / 컬럼 수 | 345390 행 / 24 열 |
| 빈칸이 있는 컬럼 | CLN-IN, TC-Z1, TC-Z2, TC-Z3, TC-Z4, TC-Z5, TC-Z6, OP-Z3, CP-Z4M, TC-Z7, TC-Z8, TC-Z9, OP-Z7, OP-Z8, TC-OUT1, TC-OUT2 |
| 빈칸 비율이 가장 높은 컬럼 | OP-Z3 (0.07 %) |
| 값이 계속 똑같은 컬럼 | 없음 |
| 완전히 똑같은 중복 행 | 0 개 |
| 같은 시각(MEAS_DT)이 두 번 이상 나온 경우 | 0 개 |


